In [1]:
import torch

# simple one-layer nn
x = torch.ones(5)  # input tensor
y = torch.zeros(3)  # expected output
w = torch.randn(5, 3, requires_grad=True)
b = torch.randn(3, requires_grad=True)
z = torch.matmul(x, w) + b
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)

In [2]:
print(f"gradient function for z = {z.grad_fn}")
print(f"gradient function for loss = {loss.grad_fn}")

gradient function for z = <AddBackward0 object at 0x1133a6f80>
gradient function for loss = <BinaryCrossEntropyWithLogitsBackward0 object at 0x1133a68c0>


In [ ]:
# compute gradients
loss.backward()
print(w.grad)
print(b.grad)

tensor([[0.0829, 0.2719, 0.0249],
        [0.0829, 0.2719, 0.0249],
        [0.0829, 0.2719, 0.0249],
        [0.0829, 0.2719, 0.0249],
        [0.0829, 0.2719, 0.0249]])
tensor([0.0829, 0.2719, 0.0249])


In [9]:
# Disable gradient tracking
print(z.requires_grad)
with torch.no_grad():
    z = torch.matmul(x, w) + b
print(z.requires_grad)

True
False


In [10]:
# another way to disable gradient tracking
z_det = z.detach()
print(z_det.requires_grad)

False


- backwardはスカラーに対して呼ばないとエラーが出る
- スカラーじゃない出力に対しては、ヤコビアン積を求める
- torhc.ones_like(out)を初期値として、backwardを計算する

In [ ]:
# when output is not scalar
inp = torch.eye(4, 5, requires_grad=True)  # identity matrix(単位行列)
out = (inp + 1).pow(2).t()  # (x+1)**2
out.backward(torch.ones_like(out), retain_graph=True)  # compute jacobian products
print(f"First call\n{inp.grad}")  # 2(x+1)
out.backward(torch.ones_like(out), retain_graph=True)
print(f"Second call\n{inp.grad}")  # 2(x+1) + 2(x+1) (accumulate!)
inp.grad.zero_()  # reset gradient to zero
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nCall after zeroing gradients\n{inp.grad}")

First call
tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])
Second call
tensor([[8., 4., 4., 4., 4.],
        [4., 8., 4., 4., 4.],
        [4., 4., 8., 4., 4.],
        [4., 4., 4., 8., 4.]])

Call after zeroing gradients
tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])


# Additional problems

問題 1: 基本的な勾配計算 (Basic Gradients)
数式 $z = 3x^2 + 2y^3$ が与えられています。 
$x = 2.0$, $y = 3.0$ の地点における $x$ と $y$ のそれぞれに対する微分（勾配）を計算するためのコードを書いてください。

条件:
- $x$ と $y$ をスカラーのテンソルとして定義し、微分対象としてマークしてください。
- $z$ を $x$ と $y$ を用いて計算してください。
- 逆伝播を実行して勾配を求めてください。
- 最後に x.grad と y.grad の結果を print してください。（数学的には $\frac{\partial z}{\partial x} = 6x$, $\frac{\partial z}{\partial y} = 6y^2$ になるはずです）

In [ ]:
import torch
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)

z = 3 * x**2 + 2 * y**3
z.backward()
print(x.grad)  # 6x = 6*2 = 12
print(y.grad)  # 6y^2 = 6*3^2 = 54

tensor(12.)
tensor(54.)


問題 2: 勾配記録の停止 (Disabling Gradient Tracking)
モデルの推論（評価）時には、メモリと計算時間を節約するために「勾配の記録」をオフにする必要があります。

テンソル w = torch.tensor([1.0, 2.0, 3.0], requires_grad=True) があります。 y = w * 2 という計算を行いますが、y に勾配の履歴（grad_fn）が残らないようにする ためのコードを 2つの異なる方法 で書いてください。

In [ ]:
import torch
w = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = w*2
print(y.grad_fn)

with torch.no_grad():
    y = w*2
print(y.grad_fn)

y2 = y.detach()
print(y2.grad_fn)

None
None


問題 3: 出力がスカラーじゃない場合の逆伝播 (ヤコビアン積)


テンソル x = torch.tensor([2.0, 3.0, 4.0], requires_grad=True) があります。 この要素をそれぞれ3乗する計算 y = x ** 3 を行います。出力 y はベクトル（3つの要素）になるため、そのまま y.backward() と書くとエラーになります。

条件: y.backward(...) に適切な重みベクトル（すべて1のテンソル） を渡して逆伝播を実行し、x.grad を表示させてください。

In [ ]:
import torch
x = torch.tensor([2.0, 3.0, 4.0], requires_grad=True)
y = x**3
y.backward(torch.ones_like(y), retain_graph=True)  # loss = y.sum(), loss.backward()と同義。要は要素ごとに微分して足している
print(x.grad)

tensor([12., 27., 48.])
